In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_moadel import LogisticRegression
from sklearn.pipeline import make_pipeline

In [1]:
# 1. Read in the dataset directly from the UCI website using pandas
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data', header=None)

# 2. Assign features to X and labels to y, then encode labels
X = df.loc[:, 2:].values
y = df.loc[:, 1].values

le = LabelEncoder()
y = le.fit_transform(y)

# Double-check the mapping: 'M' -> 1, 'B' -> 0
print(le.classes_)
print(le.transform(['M', 'B']))

# 3. Divide the dataset into training and test splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=1
)

# 4. Chain StandardScaler, PCA, and LogisticRegression objects in a pipeline
pipe_lr = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression()
)

['B' 'M']
[1 0]


# Using k-fold cross-validation to assess model performance

To obtain reliable estimates of a model's generalization performance on unseen data, we use cross-validation techniques like holdout cross-validation and k-fold cross-validation.

## The Holdout Method

* The holdout method is a classic approach that splits an initial dataset into separate training and test datasets.
* However, during model selection (tuning hyperparameters to find optimal values), reusing the same test dataset can cause the model to overfit.
* A better practice is to split the data into three parts: a training dataset (to fit models), a validation dataset (for model selection/tuning), and a test dataset (to estimate final generalization performance).
* A major disadvantage of the holdout method is that the performance estimate is highly sensitive to how the data is partitioned into training and validation subsets.

<p align="center">
  <img src="https://raw.githubusercontent.com/rasbt/machine-learning-book/main/ch06/figures/06_02.png" width="600">
</p>

## K-fold Cross-Validation

* K-fold cross-validation is a more robust resampling technique where the training dataset is randomly split into $k$ folds without replacement.
* The procedure is repeated $k$ times: $k-1$ folds are used for model training, and 1 fold is used as the test fold for performance evaluation.
* The estimated performances ($E_i$) for each fold are used to calculate the average performance ($E$) of the model. For instance, if $k=10$, the formula is:

$$E = \frac{1}{10} \sum_{i=1}^{10} E_i$$


* Once satisfactory hyperparameter values are found, the final model is retrained on the complete training dataset to obtain a final performance estimate on the independent test set.
* A standard recommended value is $k=10$, which empirical evidence suggests provides the best tradeoff between bias and variance.
* For very small datasets, **Leave-one-out cross-validation (LOOCV)** is recommended, where the number of folds equals the number of training examples ($k = n$).

<p align="center">
  <img src="https://raw.githubusercontent.com/rasbt/machine-learning-book/main/ch06/figures/06_03.png" width="600">
</p>

## Stratified k-fold cross-validation via scikit-learn

An improvement over standard k-fold cross-validation is stratified k-fold cross-validation. In stratified cross-validation, the class label proportions are preserved in each fold to ensure they represent the overall class proportions in the training dataset.

Here is how you can explicitly loop through the $k$ folds using `StratifiedKFold` in scikit-learn:

In [2]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=10).split(X_train, y_train)
scores = []

for k, (train, test) in enumerate(kfold):
    pipe_lr.fit(X_train[train], y_train[train])
    score = pipe_lr.score(X_train[test], y_train[test])
    scores.append(score)
    print(f'Fold: {k+1:02d}, '
          f'Class distr.: {np.bincount(y_train[train])}, '
          f'Acc.: {score:.3f}')

mean_acc = np.mean(scores)
std_acc = np.std(scores)
print(f'\nCV accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')

Fold: 01, Class distr.: [256 153], Acc.: 0.935
Fold: 02, Class distr.: [256 153], Acc.: 0.935
Fold: 03, Class distr.: [256 153], Acc.: 0.957
Fold: 04, Class distr.: [256 153], Acc.: 0.957
Fold: 05, Class distr.: [256 153], Acc.: 0.935
Fold: 06, Class distr.: [257 153], Acc.: 0.956
Fold: 07, Class distr.: [257 153], Acc.: 0.978
Fold: 08, Class distr.: [257 153], Acc.: 0.933
Fold: 09, Class distr.: [257 153], Acc.: 0.956
Fold: 10, Class distr.: [257 153], Acc.: 0.956

CV accuracy: 0.950 +/- 0.014


### Using `cross_val_score`

Scikit-learn implements a `cross_val_score` function that evaluates the model using stratified k-fold cross-validation automatically, which is much less verbose. You can also use the `n_jobs` parameter to distribute evaluation across multiple CPUs.

In [3]:
from sklearn.model_selection import cross_val_score

# Set n_jobs=-1 to use all available CPUs for parallel computation
scores = cross_val_score(estimator=pipe_lr,
                         X=X_train,
                         y=y_train,
                         cv=10,
                         n_jobs=1)

print(f'CV accuracy scores: {scores}')
print(f'CV accuracy: {np.mean(scores):.3f} +/- {np.std(scores):.3f}')

CV accuracy scores: [0.93478261 0.93478261 0.95652174 0.95652174 0.93478261 0.95555556
 0.97777778 0.93333333 0.95555556 0.95555556]
CV accuracy: 0.950 +/- 0.014
